# Pandas Avanzado

Este notebook cubre operaciones avanzadas en pandas para análisis y transformación de datos.

## Prerrequisitos
Se asume que ya viste:
1. Series en pandas
2. Indexación en DataFrames
3. Nuevas columnas
4. Eliminar columnas
5. Seleccionar filas y columnas
6. Selección condicional
7. `reset_index`
8. `groupby`
9. `pivot_table`
10. `concat`
11. `merge`
12. Casting
13. Manejo de `NaN`
14. Valores únicos
15. Duplicados
16. Outliers
17. Estadística descriptiva
18. Leer y guardar datos


## Objetivos de este notebook

En este notebook veremos:

1. `assign`, `pipe` y encadenamiento de operaciones
2. `query` y `eval`
3. Ordenamiento avanzado y ranking
4. Funciones ventana con `transform`
5. `map`, `replace`, `where` y `mask`
6. Manejo avanzado de fechas
7. Strings con `.str`
8. `apply` vs alternativas vectorizadas
9. `explode`
10. MultiIndex
11. Agregaciones múltiples
12. Reshape con `melt`, `stack`, `unstack`
13. Joins avanzados: `merge_asof` y `merge_ordered`
14. Validaciones y calidad de datos
15. Ejercicio integrador


In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 120)
pd.set_option('display.max_colwidth', 120)

## 1. Creamos una base de ejemplo

In [ ]:
ventas = pd.DataFrame({
    'cliente_id': [101, 101, 102, 102, 103, 103, 104, 104],
    'ciudad': ['Bogotá', 'Bogotá', 'Medellín', 'Medellín', 'Cali', 'Cali', 'Barranquilla', 'Barranquilla'],
    'segmento': ['Pyme', 'Pyme', 'Corp', 'Corp', 'Pyme', 'Pyme', 'Corp', 'Corp'],
    'fecha': pd.to_datetime(['2025-01-05', '2025-02-10', '2025-01-15', '2025-03-20', '2025-01-08', '2025-02-25', '2025-01-10', '2025-03-01']),
    'producto': ['A', 'B', 'A', 'C', 'B', 'C', 'A', 'B'],
    'monto': [1200, 1800, 2500, 3200, 900, 1500, 2700, 2100],
    'unidades': [3, 4, 5, 6, 2, 3, 5, 4],
    'canal': ['Web', 'Oficina', 'Web', 'Aliado', 'Web', 'Oficina', 'Aliado', 'Web'],
    'tags': [['nuevo', 'promo'], ['upsell'], ['vip'], ['vip', 'renovacion'], ['promo'], ['nuevo'], ['vip'], ['promo', 'crosssell']]
})

ventas

## 2. Encadenamiento de operaciones con `assign` y `pipe`

Esto ayuda a escribir transformaciones más limpias y legibles.

In [ ]:
resultado = (
    ventas
    .assign(
        precio_unitario=lambda df: df['monto'] / df['unidades'],
        mes=lambda df: df['fecha'].dt.to_period('M').astype(str)
    )
)

resultado

### Ejemplo con `pipe`

In [ ]:
def filtrar_pymes(df):
    return df[df['segmento'] == 'Pyme']

def agregar_ticket_promedio(df):
    return df.assign(ticket_promedio=df['monto'] / df['unidades'])

ventas.pipe(filtrar_pymes).pipe(agregar_ticket_promedio)

## 3. `query` y `eval`

Sirven para escribir filtros y expresiones de manera más compacta.

In [ ]:
ventas.query("segmento == 'Corp' and monto > 2500")

In [ ]:
ventas.eval("ingreso_por_unidad = monto / unidades")

## 4. Ordenamiento avanzado y ranking

In [ ]:
ventas.sort_values(['segmento', 'monto'], ascending=[True, False])

In [ ]:
ventas.assign(
    rank_monto_segmento=ventas.groupby('segmento')['monto'].rank(method='dense', ascending=False)
)

## 5. `transform`: métricas por grupo sin perder el detalle

Muy útil cuando quieres calcular una métrica agregada y mantener cada fila original.

In [ ]:
ventas.assign(
    promedio_segmento=ventas.groupby('segmento')['monto'].transform('mean'),
    share_vs_segmento=ventas['monto'] / ventas.groupby('segmento')['monto'].transform('sum')
)

## 6. `map`, `replace`, `where` y `mask`

In [ ]:
mapa_segmento = {'Pyme': 'Pequeña/Mediana', 'Corp': 'Corporativo'}

ventas['segmento'].map(mapa_segmento)

In [ ]:
ventas['canal'].replace({'Web': 'Digital', 'Oficina': 'Presencial', 'Aliado': 'Partners'})

In [ ]:
ventas.assign(
    monto_capado=ventas['monto'].where(ventas['monto'] <= 2500, 2500),
    monto_solo_altos=ventas['monto'].mask(ventas['monto'] < 2000, np.nan)
)

## 7. Fechas: componentes, offsets y resample

In [ ]:
ventas.assign(
    anio=ventas['fecha'].dt.year,
    mes=ventas['fecha'].dt.month,
    dia_semana=ventas['fecha'].dt.day_name()
)

In [ ]:
ventas.set_index('fecha').resample('M')['monto'].sum()

## 8. Operaciones con strings usando `.str`

In [ ]:
clientes = pd.DataFrame({
    'nombre': [' Ana Pérez ', 'juan gomez', 'MARIA LOPEZ'],
    'email': ['ana@test.com', 'juan@test.com', 'maria@test.com']
})

clientes.assign(
    nombre_limpio=clientes['nombre'].str.strip().str.title(),
    dominio=clientes['email'].str.split('@').str[1]
)

## 9. `apply` vs alternativas vectorizadas

Regla práctica: si puedes resolverlo con operaciones vectorizadas, mejor que con `apply`.

In [ ]:
# Menos eficiente
ventas['tipo_monto_apply'] = ventas['monto'].apply(lambda x: 'alto' if x >= 2000 else 'bajo')

# Más idiomático y escalable
ventas['tipo_monto_npwhere'] = np.where(ventas['monto'] >= 2000, 'alto', 'bajo')

ventas[['monto', 'tipo_monto_apply', 'tipo_monto_npwhere']]

## 10. Expandir listas con `explode`

In [ ]:
ventas[['cliente_id', 'producto', 'tags']].explode('tags')

## 11. MultiIndex

Útil cuando trabajas con varias dimensiones jerárquicas.

In [ ]:
multi = ventas.groupby(['segmento', 'canal'])['monto'].agg(['sum', 'mean', 'count'])
multi

In [ ]:
multi.loc[('Corp', 'Aliado')]

## 12. Agregaciones múltiples con `agg`

In [ ]:
ventas.groupby('segmento').agg(
    monto_total=('monto', 'sum'),
    monto_promedio=('monto', 'mean'),
    unidades_max=('unidades', 'max'),
    n_clientes=('cliente_id', 'nunique')
)

## 13. Reshape con `melt`, `stack` y `unstack`

In [ ]:
resumen = ventas.groupby('segmento')[['monto', 'unidades']].sum().reset_index()
resumen

In [ ]:
resumen_largo = resumen.melt(id_vars='segmento', var_name='metrica', value_name='valor')
resumen_largo

In [ ]:
tabla = ventas.groupby(['segmento', 'canal'])['monto'].sum()
tabla

In [ ]:
tabla.unstack(fill_value=0)

## 14. Joins avanzados: `merge_asof`

Muy útil para unir por la fecha más cercana hacia atrás o hacia adelante.

In [ ]:
transacciones = pd.DataFrame({
    'fecha': pd.to_datetime(['2025-01-05', '2025-01-12', '2025-02-10', '2025-03-01']),
    'cliente_id': [101, 102, 103, 104],
    'monto': [1200, 2400, 1500, 2100]
}).sort_values('fecha')

tasas = pd.DataFrame({
    'fecha': pd.to_datetime(['2025-01-01', '2025-02-01', '2025-03-01']),
    'tasa': [0.015, 0.017, 0.016]
}).sort_values('fecha')

pd.merge_asof(transacciones, tasas, on='fecha', direction='backward')

## 15. Validaciones y calidad de datos

In [ ]:
print('Filas duplicadas:', ventas.duplicated().sum())
print('Montos nulos:', ventas['monto'].isna().sum())
print('Fechas futuras:', (ventas['fecha'] > pd.Timestamp.today()).sum())
print('Montos negativos:', (ventas['monto'] < 0).sum())

## 16. Ejercicio integrador

Objetivo:

1. Crear `precio_unitario`
2. Crear `mes`
3. Calcular el ranking de `monto` dentro de cada `segmento`
4. Calcular el share del `monto` frente al total del segmento
5. Explotar la columna `tags`
6. Construir una tabla final por `segmento`, `mes` y `tag`


In [ ]:
ejercicio = (
    ventas
    .assign(
        precio_unitario=lambda df: df['monto'] / df['unidades'],
        mes=lambda df: df['fecha'].dt.to_period('M').astype(str),
        rank_monto=lambda df: df.groupby('segmento')['monto'].rank(method='dense', ascending=False),
        share_segmento=lambda df: df['monto'] / df.groupby('segmento')['monto'].transform('sum')
    )
    .explode('tags')
    .groupby(['segmento', 'mes', 'tags'], as_index=False)
    .agg(
        monto_total=('monto', 'sum'),
        clientes=('cliente_id', 'nunique'),
        ticket_promedio=('precio_unitario', 'mean')
    )
)

ejercicio

## 17. Ideas para ejercicios de práctica

1. Calcular el top 2 de productos por segmento usando ranking.
2. Encontrar qué porcentaje representa cada ciudad dentro del total de ventas.
3. Convertir una tabla ancha a larga con `melt`.
4. Crear banderas de calidad de datos.
5. Unir una tabla de eventos con la tasa más cercana usando `merge_asof`.
